# MILK10k — Step 1: Data audit + lesion-level 5-fold splits
Builds one lesion-level table (both images, metadata, per-image MONET, label, skin tone) and saves `milk10k_lesions_folds.csv`, which every later experiment should load so all models share identical folds.

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

# ---- EDIT THESE PATHS (Kaggle: /kaggle/input/<your-dataset>/...) ----
GT    = "supplements/training_gt.csv"
INP   = "supplements/training_input.csv"
SUPP  = "supplements/training_supp.csv"
IMG_DIR = "images"          # folder with ISIC_xxxxxxx.jpg
SEED, K = 42, 5

## 1. Load and merge to one row per lesion

In [ ]:
gt, inp, sup = pd.read_csv(GT), pd.read_csv(INP), pd.read_csv(SUPP)
CLASSES = [c for c in gt.columns if c != "lesion_id"]
assert (gt[CLASSES].sum(1) == 1).all(), "labels are not one-hot"
gt["label"] = gt[CLASSES].idxmax(1)
gt["y"] = gt["label"].map({c:i for i,c in enumerate(CLASSES)})

inp = inp.merge(sup, on="isic_id", how="left")
inp["mod"] = inp["image_type"].map({"clinical: close-up":"C", "dermoscopic":"D"})
assert inp.groupby("lesion_id")["mod"].apply(lambda s: sorted(s)==["C","D"]).all()

MONET = [c for c in inp.columns if c.startswith("MONET_")]
per_img = inp.pivot(index="lesion_id", columns="mod",
                    values=["isic_id","image_manipulation"] + MONET)
per_img.columns = [f"{a}_{b}" for a,b in per_img.columns]      # e.g. MONET_pigmented_C

lesion_cols = ["lesion_id","age_approx","sex","skin_tone_class","site",
               "diagnosis_full","diagnosis_confirm_type","invasion_thickness_interval"]
les = (inp[inp["mod"]=="D"][lesion_cols]
       .merge(per_img.reset_index(), on="lesion_id")
       .merge(gt[["lesion_id","label","y"]], on="lesion_id"))
les["altered_C"] = les["image_manipulation_C"].eq("altered")
les["path_C"] = IMG_DIR + "/" + les["isic_id_C"] + ".jpg"
les["path_D"] = IMG_DIR + "/" + les["isic_id_D"] + ".jpg"
print(les.shape); les.head(3).T

## 2. Audit

In [ ]:
print(les["label"].value_counts())
print("\nMissing:", les[["age_approx","sex","site","skin_tone_class"]].isna().sum().to_dict())
print("\nSkin tone (0=very dark ... 5=very light):")
print(pd.crosstab(les["label"], les["skin_tone_class"], margins=True))

In [ ]:
# MONET: per-image concepts. Which are diagnostic vs acquisition artefacts?
for m in MONET:
    a, b = les[m+"_C"].astype(float), les[m+"_D"].astype(float)
    print(f"{m:48s} C-D corr={a.corr(b):.2f}  meanC={a.mean():.2f}  meanD={b.mean():.2f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
les["label"].value_counts().plot.bar(ax=ax[0], title="Lesions per class")
les["skin_tone_class"].value_counts().sort_index().plot.bar(ax=ax[1], title="Lesions per skin tone")
plt.tight_layout(); plt.show()

## 3. Skin-tone groups + 5-fold stratified split
The lesion is the unit (both images of a lesion always share a fold). No patient ID exists, so patient-level grouping is impossible (state this as a limitation). Strata = class × tone group, falling back to class alone for rare combinations.

In [ ]:
les["tone_group"] = pd.cut(les["skin_tone_class"], [-1,2,3,5],
                          labels=["darker(0-2)","medium(3)","lighter(4-5)"]).astype(str)
strata = les["label"] + "|" + les["tone_group"]
counts = strata.value_counts()
strata = np.where(strata.map(counts) >= K, strata, les["label"])

les["fold"] = -1
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
for f, (_, te) in enumerate(skf.split(les, strata)):
    les.loc[les.index[te], "fold"] = f
assert (les["fold"] >= 0).all()
print(pd.crosstab(les["label"], les["fold"]))
print(pd.crosstab(les["tone_group"], les["fold"]))

In [ ]:
les.to_csv("milk10k_lesions_folds.csv", index=False)
print("saved", les.shape)